# Notebook 07: Multimodal Embeddings — CLIP

## Why This Matters
CLIP (Radford et al., 2021) changed what "embedding" means. Before CLIP, embeddings were unimodal: image embeddings for images, text embeddings for text. CLIP maps both into a shared space where `image("a photo of a dog")` is near `text("a photo of a dog")`. This enables zero-shot transfer, image search via text query, and cross-modal reasoning — without any task-specific training.

CLIP is now the backbone of Stable Diffusion's text conditioning, DALL-E's image retrieval, and most production image-text retrieval systems. It's the most practically useful model in this series for real applications.

## Structure
1. The CLIP objective — image-text contrastive learning (narrative)
2. Loading CLIP and encoding images and text
3. Zero-shot image classification
4. Image-text retrieval: text query → image results
5. Comparing CLIP and DINO features
6. Practical considerations: model sizes, OpenCLIP alternatives
7. Bridge to probing classifiers

## What You'll Learn
- How CLIP's contrastive objective aligns image and text representations
- How zero-shot classification works: cosine similarity against text class templates
- How to build a text→image and image→text retrieval pipeline
- When CLIP features outperform DINO features and vice versa


## Background

### The CLIP objective

CLIP is trained on 400 million (image, caption) pairs scraped from the internet. The objective is the same NT-Xent loss from SimCLR, applied across modalities:

For a batch of N (image, text) pairs:
- Encode all N images → I₁...Iₙ
- Encode all N texts → T₁...Tₙ
- Maximize cosine similarity of matched pairs (Iᵢ, Tᵢ)
- Minimize cosine similarity of all unmatched pairs (Iᵢ, Tⱼ) where i≠j

The loss is symmetric: images predict texts AND texts predict images simultaneously.

```
L = 0.5 × CE(image→text) + 0.5 × CE(text→image)
```

### Zero-shot classification

CLIP enables zero-shot classification by framing class labels as text prompts:

```python
text_prompts = ["a photo of a cat", "a photo of a dog", "a photo of a bird"]
text_features = clip.encode_text(text_prompts)     # [3, D]
image_features = clip.encode_image(image)           # [1, D]
similarities = cosine_similarity(image_features, text_features)
predicted_class = argmax(similarities)
```

No fine-tuning needed — the shared embedding space handles the classification via similarity. This is why CLIP achieves 76.2% accuracy on ImageNet with no ImageNet training.

### Architecture

CLIP uses a ViT or ResNet as the image encoder and a transformer as the text encoder. Both encode to the same dimensionality D (512 or 1024 depending on model size). A learned temperature parameter τ scales the logits.


In [ ]:
# %pip install -q openai-clip torch torchvision numpy matplotlib scikit-learn Pillow

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import clip
from torch.utils.data import DataLoader

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load CLIP ViT-B/32 — good balance of speed and quality
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

print(f"CLIP model: ViT-B/32")
print(f"Image encoder output dim: {model.visual.output_dim}")
print(f"Text encoder output dim:  {model.transformer.width}")
print(f"Embedding dim (shared):   {model.visual.output_dim}")

# Available CLIP models — larger = better quality, slower
available = clip.available_models()
print(f"\nAvailable models: {available}")

## 1. Encoding Images and Text

In [ ]:
# Load CIFAR-10 test set with CLIP's preprocessing
cifar_test = torchvision.datasets.CIFAR10(
    "/tmp/cifar10", train=False, download=True,
    transform=preprocess  # CLIP's own preprocessing — 224px, normalized for ViT
)
test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=0)

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Encode class names as text
# Prompt engineering matters — "a photo of a {class}" outperforms just "{class}"
text_prompts = [f"a photo of a {c}" for c in classes]
text_tokens  = clip.tokenize(text_prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print(f"Text features shape: {text_features.shape}  ({len(classes)} classes × embedding_dim)")
print()
print("Text prompts:")
for p in text_prompts:
    print(f"  {p}")

## 2. Zero-Shot Classification

In [ ]:
def clip_zero_shot_accuracy(model, loader, text_features, device):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            img_feats = model.encode_image(imgs)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            # Similarity to each class text embedding
            logits = (img_feats @ text_features.T) * model.logit_scale.exp()
            preds = logits.argmax(dim=-1)
            all_preds.append(preds.cpu()); all_labels.append(labels)
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return (preds == labels).mean()


print("Running CLIP zero-shot classification on CIFAR-10...")
acc = clip_zero_shot_accuracy(model, test_loader, text_features, device)
print(f"\nZero-shot accuracy: {acc:.3f}")
print("(CLIP trained on internet images, never seen CIFAR-10 labels)")
print("(ImageNet zero-shot: 76.2% with ViT-L/14 — best model)")
print()
print("Prompt matters — 'a photo of a {class}' typically outperforms just '{class}'")
print("OpenAI uses 80 prompt templates and ensembles for best results.")

In [ ]:
# Per-class breakdown
from sklearn.metrics import confusion_matrix
import seaborn as sns

all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        img_feats = model.encode_image(imgs)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        logits = (img_feats @ text_features.T) * model.logit_scale.exp()
        preds = logits.argmax(dim=-1)
        all_preds.append(preds.cpu()); all_labels_list.append(labels)

preds_np  = torch.cat(all_preds).numpy()
labels_np = torch.cat(all_labels_list).numpy()

print("Per-class accuracy:")
for i, cls in enumerate(classes):
    mask = labels_np == i
    cls_acc = (preds_np[mask] == labels_np[mask]).mean()
    bar = '█' * int(cls_acc * 20)
    print(f"  {cls:12s}: {cls_acc:.2f} {bar}")

## 3. Image-Text Retrieval

In [ ]:
# Encode a gallery of images once, then search with text queries
print("Encoding gallery images...")
gallery_features, gallery_labels_list = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        feats = model.encode_image(imgs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        gallery_features.append(feats.cpu()); gallery_labels_list.append(labels)

gallery_features = torch.cat(gallery_features)  # [N, D]
gallery_labels   = torch.cat(gallery_labels_list).numpy()
print(f"Gallery: {gallery_features.shape[0]} images encoded")
print()

def text_to_image_search(query: str, gallery_features, gallery_labels,
                          model, device, top_k=5):
    """Find the top-k gallery images most similar to a text query."""
    tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        q_feat = model.encode_text(tokens)
        q_feat = q_feat / q_feat.norm(dim=-1, keepdim=True)
    sims = (q_feat.cpu() @ gallery_features.T).squeeze(0).numpy()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(idx, sims[idx], classes[gallery_labels[idx]]) for idx in top_idx]


queries = [
    "a small furry animal",
    "a vehicle with wheels",
    "something that can fly",
    "an animal that lives in water",
    "a large powerful animal",
]

print("Text → Image retrieval results:")
print("-" * 60)
for q in queries:
    results = text_to_image_search(q, gallery_features, gallery_labels, model, device)
    retrieved = [f"{cls}({sim:.2f})" for _, sim, cls in results]
    print(f"  {q!r:<35}: {retrieved}")

## 4. CLIP vs DINO — When to Use Which

In [ ]:
comparison = {
    "Modalities":          ("Image + Text", "Image only"),
    "Training objective":  ("Cross-modal contrastive (image↔caption)", "Self-distillation (student→teacher)"),
    "Training data":       ("400M internet image-caption pairs", "ImageNet (1M images, no captions)"),
    "Zero-shot via text":  ("Yes — cosine sim against text prompts", "No — no text encoder"),
    "Spatial features":    ("Weaker — global CLS focus", "Strong — patch attention maps"),
    "Segmentation":        ("Poor zero-shot", "Emergent — attends to object boundaries"),
    "Text-image retrieval":("Native", "Requires separate text encoder"),
    "Linear probe (IN1k)": ("~76% ViT-B/32", "~77% ViT-S/8 (smaller model!)"),
    "Best for":            ("Multimodal tasks, open-vocab classification", "Dense prediction, segmentation, retrieval"),
}

print(f"{'Aspect':<25} {'CLIP':<45} {'DINO'}")
print("-" * 100)
for aspect, (clip_v, dino_v) in comparison.items():
    print(f"{aspect:<25} {clip_v:<45} {dino_v}")

print()
print("Rule of thumb:")
print("  Need text queries → CLIP")
print("  Need spatial understanding → DINO (or DINOv2)")
print("  Need both → use CLIP for retrieval, DINO for dense features, combine")

## 5. Practical Considerations

### Model size tradeoff

| Model | Params | ImageNet | Speed |
|-------|--------|----------|-------|
| ViT-B/32 | 151M | 63.3% | Fast |
| ViT-B/16 | 150M | 68.3% | Moderate |
| ViT-L/14 | 428M | 75.3% | Slow |
| ViT-L/14@336px | 428M | 76.2% | Slowest |

### OpenCLIP

OpenAI's CLIP weights are available via `openai-clip`. OpenCLIP (LAION) provides open-source weights trained on LAION-400M and LAION-2B:

```python
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms("ViT-H-14", pretrained="laion2b_s32b_b79k")
```

LAION-trained models often outperform original CLIP on specialized domains because the training data is more diverse.

### DINOv2

The DINO successor (Oquab et al., 2023) adds curated training data, registers, and a distillation step from a larger teacher. DINOv2 features dominate on dense tasks and are the current state-of-the-art for frozen visual features.

```python
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
```


## 6. Bridge to Probing Classifiers

You've now used CLIP and DINO features with a linear classifier for evaluation. That linear evaluation protocol — freeze the encoder, train only a linear layer on top — is called **linear probing**, and it's the standard measure of representation quality.

But probing tells us more than just "how good are the features overall." By probing *at different layers* of the network, you can ask: where in the network does semantic information live? Does color information concentrate in early layers? Does object identity emerge in later layers?

Notebook 08 covers the linear probing protocol systematically — layer-by-layer analysis of CLIP and DINO representations to understand what information lives where.


## Summary

| Concept | Key Point |
|---------|-----------|
| CLIP objective | Cross-modal NT-Xent: maximize similarity of matched (image, caption) pairs |
| Zero-shot classification | Cosine similarity between image and text class prompt embeddings |
| Shared embedding space | Images and text map to the same space — enables cross-modal retrieval |
| Prompt engineering | "a photo of a {class}" beats "{class}" — framing affects similarity scores |
| CLIP vs DINO | CLIP for multimodal/text queries; DINO for spatial/dense features |
| OpenCLIP | Open-source CLIP weights trained on LAION — often stronger on specialized domains |

**Next:** Notebook 08 — Probing Classifiers. Layer-by-layer analysis of what information lives where in CLIP and DINO networks: a principled way to understand representation quality beyond top-1 accuracy.
